# C00 — M3C3: Categorical Encoding and Feature Construction
Predict whether a customer will leave. The inputs describe the customer before the outcome; **Churn** is the known answer. **customerID** is only a label.

**Make your own copy in Colab first.** Save changes to your own notebook as you go. Keep the issued `telco_source.csv` nearby. Run the prepared cells in order; their code is folded. Edit your answers and the single AI function in C09.

Route: read real rows → represent names → record a starting result → build and check one counting rule → add five candidates → compare → explain and save.

My input examples: ___  
The answer we want to predict: ___

## C01 — Load and read five real customers
Run this prepared cell. Upload the issued CSV if asked. Read the output before continuing.
`tenure` = months with the company; `MonthlyCharges` = current monthly charge; `TotalCharges` = recorded total so far. A blank amount means unknown. `Churn` = known Yes/No answer. The final test remains closed.

In [ ]:
#@title C01 — Load issued CSV and show five training rows
import base64, hashlib, json, zlib, marshal
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

_SPLIT = json.loads(zlib.decompress(base64.b64decode('...')))
_SOURCE_SHA = '16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91'
_BASE_NUM = ['SeniorCitizen','tenure','MonthlyCharges','TotalCharges']
_CAT = ['gender','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']
_ORIGINAL_PREDICTORS = _BASE_NUM + _CAT
_SERVICES = ['PhoneService','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
FEATURE_NAMES = ['charge_per_month','monthly_charge_x_tenure','tenure_years','num_services','charge_per_service']


def _check_raw():
    assert hashlib.sha256(Path('telco_source.csv').read_bytes()).hexdigest() == _SOURCE_SHA, 'The issued raw CSV changed. Restore the original file before continuing.'
    assert raw.equals(_raw_snapshot), 'raw was changed. Reload the original source; do not edit raw.'
    assert raw.customerID.is_unique and not raw.duplicated().any(), 'Unexpected duplicate rows or IDs.'
    groups = [set(_SPLIT['ids'][k]) for k in ['train','validation','final_test']]
    assert not (groups[0]&groups[1] or groups[0]&groups[2] or groups[1]&groups[2]), 'Split IDs overlap.'
    assert set.union(*groups) == set(raw.customerID), 'Split does not match the source.'
    assert len(groups[0]) == 4225 and len(groups[1]) == 1409 and len(groups[2]) == 1409, 'Frozen split counts changed.'

def _service_count(df):
    out = pd.Series(0, index=df.index, dtype='int64')
    for col in _SERVICES:
        vals = df[col]
        if col == 'InternetService':
            allowed = {'DSL','Fiber optic','No'}
            found = set(vals.dropna().unique())
            assert found <= allowed, f'Unmapped InternetService value(s): {sorted(found-allowed)}'
            active = vals.isin(['DSL','Fiber optic'])
        else:
            allowed = {'Yes','No','No internet service','No phone service'}
            found = set(vals.dropna().unique())
            assert found <= allowed, f'Unmapped {col} value(s): {sorted(found-allowed)}'
            active = vals.eq('Yes')
        assert vals.notna().all(), f'Missing service value in {col}; inspect it instead of counting it as zero.'
        out += active.astype('int64')
    return out

def _assert_working_rows():
    _check_raw()
    indexed=raw.set_index('customerID',drop=False)
    for key,frame,target in [('train',train_df,train_y),('validation',validation_df,validation_y)]:
        expected=indexed.loc[_SPLIT['ids'][key]].copy()
        expected['TotalCharges']=pd.to_numeric(expected.TotalCharges.astype('string').str.strip(),errors='coerce')
        assert frame.equals(expected), f'{key} rows changed; reload the issued source before continuing.'
        assert target.equals(frame.Churn), f'{key} target changed; reload the issued source.'

def _pipeline(numeric,categorical):
    prep = ColumnTransformer([('num',Pipeline([('fill',SimpleImputer(strategy='median',keep_empty_features=True)),('scale',StandardScaler())]),numeric),
        ('cat',Pipeline([('fill',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical)], remainder='drop')
    return Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1000))])

_CHECKED = _BUILT = _BASELINE = _ENHANCED = _BASE_DATA = _MODEL_STATE = None

def load_source():
    global raw, _raw_snapshot, train_df, validation_df, train_y, validation_y
    global _CHECKED, _BUILT, _BASELINE, _ENHANCED, _BASE_DATA, _MODEL_STATE
    path=Path('telco_source.csv')
    if not path.exists():
        try: from google.colab import files
        except ImportError as exc: raise FileNotFoundError('Place issued telco_source.csv beside this notebook.') from exc
        print('Upload issued telco_source.csv.'); files.upload()
    assert hashlib.sha256(path.read_bytes()).hexdigest()==_SOURCE_SHA, 'Wrong or changed CSV.'
    raw=pd.read_csv(path,dtype={'TotalCharges':'string'}); _raw_snapshot=raw.copy(deep=True)
    _check_raw(); indexed=raw.set_index('customerID',drop=False)
    train_df=indexed.loc[_SPLIT['ids']['train']].copy(); validation_df=indexed.loc[_SPLIT['ids']['validation']].copy()
    for frame in [train_df,validation_df]: frame['TotalCharges']=pd.to_numeric(frame.TotalCharges.str.strip(),errors='coerce')
    train_y=train_df.Churn.copy(); validation_y=validation_df.Churn.copy()
    assert len(_ORIGINAL_PREDICTORS)==19 and set(_ORIGINAL_PREDICTORS)==set(raw)-{'customerID','Churn'}
    _CHECKED=_BUILT=_BASELINE=_ENHANCED=_BASE_DATA=_MODEL_STATE=None
    _assert_working_rows()
    print('Five real training customers. Inputs describe customers; Churn is the known answer.')
    display(train_df[['tenure','MonthlyCharges','TotalCharges','PhoneService','InternetService','PaymentMethod','Churn']].head(5).reset_index(drop=True))
    print('Training: 4225. Validation: 1409. Separate final test: 1409, kept closed.')

def _data_state():
    _assert_working_rows()
    h=hashlib.sha256()
    for frame in [train_df,validation_df]: h.update(pd.util.hash_pandas_object(frame,index=True).values.tobytes())
    return h.hexdigest()

def _state():
    h=hashlib.sha256(_data_state().encode()); h.update(marshal.dumps(chosen_feature.__code__))
    h.update(repr(chosen_feature.__defaults__).encode()); h.update(chosen_rule.encode())
    for frame in [train_df,validation_df]:
        value=_student_values(frame); h.update(pd.util.hash_pandas_object(value,index=True).values.tobytes())
    return h.hexdigest()

def _student_values(frame):
    original=frame.copy(deep=True)
    result=chosen_feature(frame.copy(deep=True))
    assert frame.equals(original), 'Input rows changed.'
    assert isinstance(result,pd.Series) and result.index.equals(frame.index), 'Return one numeric column aligned with the original rows.'
    values=pd.to_numeric(result,errors='raise').astype(float)
    expected=_service_count(frame)
    matches=np.isclose(values.to_numpy(),expected.to_numpy(),equal_nan=True)
    if not matches.all():
        position=int(np.flatnonzero(~matches)[0])
        print('Non-identifying service pattern to reproduce this mismatch. These are observed service values, not a made-up example. No ID, answer or amounts are shown.')
        display(pd.DataFrame({'service field':_SERVICES,'observed value':frame.iloc[position][_SERVICES].to_list()}))
        display(pd.DataFrame([{'expected count':int(expected.iloc[position]),'actual count':float(values.iloc[position])}]))
    assert matches.all(), 'Service count differs from the confirmed rule. Use the displayed eight-field pattern to repair only C09, then rerun C09 → C10 → C13 → C14 → C15 → C17.'
    return values

def _fit(extra=None):
    a=train_df.copy(); b=validation_df.copy(); nums=list(_BASE_NUM)
    if extra is not None:
        nums+=FEATURE_NAMES
        for c in FEATURE_NAMES: a[c]=extra['train'][c]; b[c]=extra['validation'][c]
    model=_pipeline(nums,_CAT).fit(a[nums+_CAT],train_y)
    prediction=model.predict(b[nums+_CAT]); correct=int((prediction==validation_y.to_numpy()).sum())
    return {'correct':correct,'checked':len(validation_y),'percent correct':round(100*correct/len(validation_y),2)}

def baseline():
    global _BASELINE,_BASE_DATA,_ENHANCED,_MODEL_STATE
    _BASELINE=_BASE_DATA=_ENHANCED=_MODEL_STATE=None
    token=_data_state(); _BASELINE=_fit(); _BASE_DATA=token
    display(pd.DataFrame([{'inputs':'Original 19',**_BASELINE}]))
    print('Correct / checked counts matching predictions. Percentage = correct / checked × 100.')

def prompt_feature():
    assert chosen_rule.strip(), 'Write your counting rule in the form first.'
    print('START REQUEST\nWrite Python only: one function chosen_feature(df). pandas as pd is available.')
    print('My own rule:',chosen_rule)
    print('Count active services using exactly these fields:',', '.join(_SERVICES))
    print('InternetService: DSL or Fiber optic = active; No = inactive. Other seven: Yes = active; No, No internet service, No phone service = inactive. Unknown or missing labels must raise an error.')
    print('Return one numeric pandas Series preserving df.index, so results stay with their original rows. Use only these fields. Do not use customerID or Churn, load files, change other code, inspect test rows, or train a model. Use made-up examples; for repair you may use the displayed non-identifying service pattern for a small reproduction, never IDs, Churn or full customer rows. I will run your function.\nEND REQUEST')

def service_examples():
    frame=pd.DataFrame('No',index=['worked','your prediction'],columns=_SERVICES)
    frame.loc['worked',['PhoneService','InternetService','OnlineSecurity']]=['Yes','DSL','Yes']
    frame.loc['your prediction','PhoneService']='Yes'
    counts=_service_count(frame)
    view=frame.T.reset_index().rename(columns={'index':'service field'})
    view=pd.concat([view,pd.DataFrame([{'service field':'TOTAL ACTIVE','worked':counts.loc['worked'],'your prediction':counts.loc['your prediction']}])],ignore_index=True)
    display(view)

def check_function():
    global _CHECKED,_BUILT,_ENHANCED,_MODEL_STATE
    _CHECKED=_BUILT=_ENHANCED=_MODEL_STATE=None
    assert chosen_rule.strip(), 'Write your own C08 rule.'
    rows=[]
    for label,expected in [('ordinary',3),('no services',0),('unknown label','error'),('missing label','error')]:
        frame=pd.DataFrame('No',index=['made_up'],columns=_SERVICES)
        if label=='ordinary': frame.loc['made_up',['PhoneService','InternetService','OnlineSecurity']]=['Yes','DSL','Yes']
        if label=='unknown label': frame.loc['made_up','OnlineSecurity']='Maybe'
        if label=='missing label': frame.loc['made_up','OnlineSecurity']=None
        try:
            result=chosen_feature(frame.copy(deep=True))
        except Exception as exc:
            actual='error' if expected=='error' and isinstance(exc,(ValueError,AssertionError,TypeError,KeyError)) else type(exc).__name__+': '+str(exc)
        else:
            if expected=='error': actual='accepted (must stop)'
            else:
                try:
                    assert isinstance(result,pd.Series) and result.index.equals(frame.index), 'Return one numeric column aligned with original rows.'
                    actual=float(pd.to_numeric(result,errors='raise').iloc[0])
                except Exception as exc: actual=type(exc).__name__+': '+str(exc)
        rows.append({'case':label,**frame[['PhoneService','InternetService','OnlineSecurity']].iloc[0].to_dict(),'expected':expected,'actual':actual})
    print('Made-up checks. The other five service fields are No in every case.')
    display(pd.DataFrame(rows))
    assert all(r['actual']==r['expected'] for r in rows), 'Check failed. Repair only C09, then rerun C09 → C10 → C13 → C14 → C15 → C17.'
    _CHECKED=_state(); print('Prepared cases passed for your current function. Add your own explanation below.')

def _divide(a,b):
    a=pd.to_numeric(a,errors='coerce').astype(float); b=pd.to_numeric(b,errors='coerce').astype(float)
    result=a.div(b).where(b.ne(0),0)
    return result.where(a.notna() & b.notna())

def ratio_examples():
    f=pd.DataFrame({'TotalCharges':[240,120,40,np.nan,40,np.nan],'tenure':[12,6,0,12,np.nan,0]})
    f['charge_per_month']=_divide(f.TotalCharges,f.tenure); display(f)
    print('TotalCharges is recorded history. MonthlyCharges × tenure is a proxy using today’s monthly charge. The monthly rate can change, so today’s rate × months need not match recorded history.')

def _features(frame):
    n=_student_values(frame); result=pd.DataFrame(index=frame.index)
    result['charge_per_month']=_divide(frame.TotalCharges,frame.tenure)
    result['monthly_charge_x_tenure']=frame.MonthlyCharges*frame.tenure
    result['tenure_years']=frame.tenure/12
    result['num_services']=n; result['charge_per_service']=_divide(frame.MonthlyCharges,n)
    return result[FEATURE_NAMES]

def build_features():
    global _BUILT,_ENHANCED,_MODEL_STATE
    _BUILT=_ENHANCED=_MODEL_STATE=None
    assert _CHECKED==_state(), 'Current function needs fresh C10 checks.'
    _BUILT={'train':_features(train_df),'validation':_features(validation_df)}
    print('Five candidate columns built. Run C14 to see five training rows and prepared formula checks.')

def preview_features(show=True):
    assert _CHECKED==_state() and _BUILT is not None, 'Rerun C10 and C13.'
    synthetic=pd.DataFrame('No',index=['control'],columns=_SERVICES)
    synthetic.loc['control',['PhoneService','InternetService','OnlineSecurity']]=['Yes','DSL','Yes']
    synthetic['tenure']=12.; synthetic['MonthlyCharges']=20.; synthetic['TotalCharges']=240.
    actual=_features(synthetic).iloc[0]; expected=[20.,240.,1.,3.,20/3]
    assert np.allclose(actual.to_numpy(),expected)
    assert np.allclose(_divide(pd.Series([40,np.nan,40,np.nan]),pd.Series([0,0,np.nan,12])),[0,np.nan,np.nan,np.nan],equal_nan=True)
    for key,frame in [('train',train_df),('validation',validation_df)]: assert _BUILT[key].equals(_features(frame)), 'Feature table changed; rerun C13.'
    if not show: return
    print('Five REAL training rows. Match preview row 0–4 across these two tables.')
    inputs=train_df[['tenure','MonthlyCharges','TotalCharges']].head(5).reset_index(drop=True)
    inputs.insert(0,'preview row',range(5)); display(inputs)
    features=_BUILT['train'].head(5).reset_index(drop=True)
    features.insert(0,'preview row',range(5)); display(features)
    print('REAL preview row 0: the eight service inputs, shown vertically for a hand count. Compare with num_services in row 0 above.')
    display(pd.DataFrame({'service field':_SERVICES,'row 0 value':train_df.iloc[0][_SERVICES].to_list()}))
    display(pd.DataFrame({'feature':FEATURE_NAMES,'control expected':expected,'actual':actual.to_numpy(),'check':['passed']*5}))
    print('Known amount / zero → 0; missing required input → missing, including missing amount / zero. These prepared cases passed.')

def compare_models():
    global _ENHANCED,_MODEL_STATE
    _ENHANCED=_MODEL_STATE=None
    assert _BASELINE is not None and _BASE_DATA==_data_state(), 'Run C05 for the current data first.'
    assert _CHECKED==_state() and _BUILT is not None, 'Rerun C09 → C10 → C13 → C14 before C15.'
    preview_features(show=False)
    _ENHANCED=_fit(_BUILT); _MODEL_STATE=_state()
    display(pd.DataFrame([{'inputs':'Original 19',**_BASELINE},{'inputs':'Original + five',**_ENHANCED}]))
    print('Same training customers, same prepared model and same validation customers. Final test stays closed.')
    print('Change in correct:',_ENHANCED['correct']-_BASELINE['correct'])

def save_guard():
    assert _BASE_DATA==_data_state() and _MODEL_STATE==_CHECKED==_state(), 'Results are missing or stale. Rerun C05 → C09 → C10 → C13 → C14 → C15 → C17.'
    assert _ENHANCED is not None and _BUILT is not None
    for key,frame in [('train',train_df),('validation',validation_df)]: assert _BUILT[key].equals(_features(frame)), 'Features changed; rebuild and compare.'
    print('Current code, source, checks and results are ready. This check does not read your Markdown discussion or save your notebook.')
    print('Complete C16 in your own words. Save/download the executed .ipynb. Reopen it and check C09 code, C15 outputs and your C16 text. Follow the announced LMS route and deadline.')

load_source()

## C02 — Read one customer
Choose one of the five displayed training rows.

In my own words, this customer: ___  
One input known before the outcome: ___  
Why would using the answer itself as an input be unfair? ___  
What should I check before using a new field: could I know it when making the prediction? My example: ___

## C03 — Names need a representation
`PaymentMethod` contains names. Coding Bank transfer = 0, Credit card = 1, Electronic check = 2 would invent an order and distances. These payment names are not amounts. By contrast, **Low < Medium < High** really describes an order.

We can ask one Yes/No question per name. **One-hot encoding** records 1 for the matching name and 0 for the others.

| PaymentMethod | Is Bank transfer? | Is Credit card? | Is Electronic check? |
|---|---:|---:|---:|
| Bank transfer | 1 | 0 | 0 |
| Credit card | ___ | ___ | ___ |
| Electronic check | ___ | ___ | ___ |

Write your predictions for the last two rows **before looking at the answer matrix on Slide 3 and before running C04**.

## C04 — Compare your representation
Run the prepared example. These are made-up labels, not customer data. The example shortens payment labels for readability; real CSV labels may include an automatic-payment suffix. The original CSV is unchanged.

In [ ]:
#@title C04 — See three names beside their numeric representation
toy=pd.DataFrame({'PaymentMethod':['Bank transfer','Credit card','Electronic check']})
encoder=OneHotEncoder(categories=[toy.PaymentMethod.tolist()],sparse_output=False)
encoded=encoder.fit_transform(toy[['PaymentMethod']]).astype(int)
display(pd.concat([toy,pd.DataFrame(encoded,columns=['Is Bank transfer?','Is Credit card?','Is Electronic check?'])],axis=1))

My prediction matched / differed here: ___  
Why does 1 not mean “better payment method”? ___

## C05 — Record the starting result
Run the prepared model using all 19 original inputs. It learns from training customers and predicts answers for the separate validation customers. The prepared setup also turns actual text categories such as PaymentMethod into numeric one-hot columns, as in C04. You do not need to do it by hand. The setup learns its preparation rules from training rows only. We keep the same model and customers for the later comparison.

**Correct** counts predictions matching the known answers. **Checked** counts validation customers. Percentage correct = correct ÷ checked × 100.

In [ ]:
#@title C05 — Run the original-input model
baseline()

My starting result: ___ correct out of ___ checked (___%).  
In my words, “correct” means: ___  
Does this result prove it works for every future customer? Why? ___

## C06 — Ask a new question: how many services are active?
**Made-up worked customer:** PhoneService = Yes, InternetService = DSL, OnlineSecurity = Yes; the other five service fields = No. Count: **1 + 1 + 1 = 3**.

**Why try this for the model?** A count of 1 versus 6 describes the overall number of subscriptions in one column. This may make a pattern more convenient for the model to use. The count does not say **which** services are active, so we keep the original service fields too. Whether the extra count helps is a question we will check, not a promise.

Use exactly eight fields: `PhoneService`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`.

- InternetService: **DSL / Fiber optic** are active; **No** is inactive.
- Other seven fields: **Yes** is active; **No / No internet service / No phone service** are inactive.
- An unknown or missing service label must stop the calculation with an error.

**Predict before C07:** another made-up customer has PhoneService = Yes and all other seven fields = No. My count: ___

In [ ]:
#@title C07 — Check both made-up service counts
service_examples()

My prediction matched / differed: ___  
The new column `num_services` tells me: ___

## C08 — Explain your rule, then ask AI
Write your counting rule in the form below. If no form field appears, show the short cell code and put your own sentence between the quotes after `chosen_rule =`. Run the cell and copy the **whole printed request**, including the mapping, to your usual AI assistant. No real customer data is needed. The AI writes one function; you run and check it.

In [ ]:
#@title C08 — Write my rule and print the complete request
chosen_rule = '' #@param {type:"string"}
prompt_feature()

## C09 — Paste the one AI function
Replace the function below with the AI's Python function `chosen_feature(df)`. Paste Python only, without Markdown fences. Run C09, then C10.

The function returns one numeric column, keeping each result with its original row. You do not need to edit the prepared checks.

**If C09 shows an error, stop.** Fix and rerun C09 before C10. An earlier green output does not check a failed edit.

In [ ]:
def chosen_feature(df):
    # Replace this function with the Python function from your AI request.
    pass

## C10 — Run the prepared checks on your actual function
Run this cell. Read the inputs, expected count and actual count for an ordinary customer, no services, and an unknown label. A missing label is also checked.

**If a check fails:** keep the original task. Send AI your **C09 function**, the **complete current C08 request**, the failing input, expected result and exact error/output. Ask it to repair **only chosen_feature(df)**. Do not ask it to change the checks, files, split or model. Replace only C09.

If the small checks pass but the real-row count stops, copy only the displayed **eight-field non-identifying service pattern**, your own function, the complete C08 request and the exact error. Do not copy IDs, Churn, amounts or a full customer row. Ask AI to repair only C09.

After a repair run **C09 → C10 → C13 → C14 → C15 → C17**. **After a real runtime restart OR changed data**, first save your own notebook and restore your existing C09 code, C08 rule and written answers, then rerun **C01 → C04 → C05 → C07 → C08 → C09 → C10 → C12 → C13 → C14 → C15 → C17** and review your answers. If AI is unavailable, ask the mentor for a clearly labelled mentor-provided function, or mark this step pending; a prepared demonstration is not your own completed run.

In [ ]:
#@title C10 — Check my function on prepared inputs
check_function()

An ordinary case I checked: ___  
Expected / actual: ___ / ___  
Why should an unknown label stop instead of silently count as zero? ___  
Code source: AI / mentor-provided / pending: ___

## C11 — A second question: historical charge per month
Made-up example: total recorded charge **240** over **12 months** gives **240 ÷ 12 = 20 per month**. This historical average may differ from today's MonthlyCharges.

**Before C12:** total 120 over 6 months. My prediction: ___ per month.

Exercise rule: known amount divided by known zero gives 0. A missing required input stays missing, including a missing amount with zero months. This is a calculation rule for this exercise, not a claim that an unknown bill is zero.

In [ ]:
#@title C12 — Check the arithmetic and missing-input policy
ratio_examples()

My prediction matched / differed: ___  
Why MAY recorded total differ from current monthly charge × months? ___

## C13 — Add five candidate descriptions
Your checked function supplies `num_services`; the other four rules are prepared. Run only after C10 passes.

| Group | Feature | Rule and meaning |
|---|---|---|
| Count | num_services | Count the eight active services |
| Ratio | charge_per_month | TotalCharges ÷ tenure: recorded average per month |
| Ratio | charge_per_service | MonthlyCharges ÷ num_services: current charge per active service |
| Other calculation | monthly_charge_x_tenure | MonthlyCharges × tenure: proxy total using today's rate |
| Other calculation | tenure_years | tenure ÷ 12: the same duration in years |

These reuse existing information. A new column is a candidate, not a guaranteed improvement. The division policy from C11 applies to both ratios.

For `charge_per_service`, if `num_services` is zero, the output 0 is **only the exercise policy**, not an actual price per service. We cannot calculate an average over zero services.

In [ ]:
#@title C13 — Build all five columns
build_features()

## C14 — Read five real rows and check one yourself
Run the preview and prepared formula checks. Read one row across from the original inputs to the five new columns.

In [ ]:
#@title C14 — Preview five training rows and check all five formulas
preview_features()

My chosen row and feature: ___  
Inputs I used: ___  
My hand calculation: ___  
Displayed result: ___  
In plain words, this result describes: ___

## C15 — Change the inputs; keep the check the same
**Before running:** I predict original + five will give more / the same / fewer correct answers, because: ___

Run the prepared comparison. It uses the same model, training-only preprocessing, customers and validation check as C05. The separate final test stays closed. If C09 changes, its old results cannot count as current: rerun C10 → C13 → C14 → C15; rerun C05 too if data changes.

In [ ]:
#@title C15 — Compare original inputs with original + five
compare_models()

Actual result: original ___ / ___; original + five ___ / ___.  
Change in correct predictions: ___. My prediction matched / differed: ___.  
What this single comparison can and cannot show: ___

## C16 — My own one-page discussion
Write in your own words using your current outputs. Do not copy the mentor demonstration.

1. What did I predict before adding columns, and why? ___
2. Choose one new column: what question does it answer, using which existing inputs? ___
3. What hand calculation and prepared check give evidence that the calculation is right? Include actual inputs, expected and observed values. ___
4. Original versus original + five: give current correct/checked counts, explain what changed, and one limit of the conclusion. ___

**Transfer:** a shop records 3 visits last month and 2 this month. Propose a new column and its value. Why might it still fail to improve a prediction? ___

## C17 — Ready check, then save and reopen
Run the prepared readiness check. It checks current code/data/results; it cannot check your own explanation.

1. Finish C16 and confirm the latest C10, C14 and C15 outputs are visible.
2. Save your Colab copy, then download the executed **.ipynb** using File → Download → Download .ipynb.
3. Reopen the downloaded file. Check your C09 function, C15 output and own C16 text are present. Follow the announced LMS submission route and deadline.

If ready check fails, follow the nearby C10 repair route. Do not replace a failed check with an old screenshot or claim that saving proves understanding.

In [ ]:
#@title C17 — Check current work before manually saving
save_guard()